# Medical Framework — Unified Pipeline
Single `imblearn.Pipeline` searched by `GridSearchCV` over a pre-sampled
Latin Hypercube grid (200 candidates with per-family quotas). Each step is
one of the custom transformers from the `.py` modules; the candidate grid
swaps each step out per draw.

In [1]:
import os
import shutil
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from imblearn.pipeline import Pipeline
from imblearn import FunctionSampler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve, f1_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from loader import load_data

from clean_data import MeanImputer, KNNImputerWrapper, IterativeModelImputer, DropRowsImputer, ClassMeanImputer, InterpolateImputer
from tame_outlier import OutlierTamer
from normalization import RobustScalerNorm, ZScoreNormalizationNorm
from feature_selection import SelectKBestFilter, TreeBasedSelection
from balance import SMOTESampler, BorderlineSMOTESampler
from model_training import (
    LogisticRegressionEstimator, RandomForestEstimator,
    XGBoostEstimator, LightGBMEstimator, CatBoostEstimator,
)

# imblearn-native no-op sampler — replaces the custom IdentitySampler whose
# `_sampling_type = "bypass"` was the most likely culprit for the NaN-everywhere
# CV scores. FunctionSampler with a pass-through func is officially supported.
def _identity(X, y):
    return X, y

def make_identity_sampler():
    return FunctionSampler(func=_identity, validate=False)

# Clear any stale joblib pipeline cache from previous failed runs. A poisoned
# cache combined with n_jobs>1 was the second suspect for the NaN scores.
shutil.rmtree('./cache', ignore_errors=True)


## Load & split

In [2]:
path = './Final/data'
typeData = 'csv'
y_column = 'CVD.event'

X, Y, all_mappings, y_mappings = load_data(path=f'{path}.{typeData}', y_column=y_column)
X = X.astype('float32')

print(f'X shape : {X.shape}')
print(f'Classes : {pd.Series(Y).value_counts().to_dict()}')

x_train, x_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y,
)

# Imbalance ratio used to seed scale_pos_weight searches downstream.
neg, pos = (np.array(y_train) == 0).sum(), (np.array(y_train) == 1).sum()
spw_base = float(neg) / float(max(pos, 1))
print(f'neg/pos in train: {neg}/{pos}  ->  scale_pos_weight base ~ {spw_base:.2f}')

X shape : (7433, 55)
Classes : {0: 6596, 1: 837}
neg/pos in train: 5276/670  ->  scale_pos_weight base ~ 7.87


## Build the pipeline
Six stages: imputation → outlier flags → normalization → feature selection → balancing → classifier. The starting values are placeholders — `RandomizedSearchCV` swaps each step out below.

In [3]:
from sklearn.base import is_classifier

pipeline = Pipeline(steps=[
    ('imputer',    MeanImputer()),
    ('tamer',      OutlierTamer()),
    ('normalizer', ZScoreNormalizationNorm()),
    ('selector',   SelectKBestFilter(k=20)),
    ('balancer',   make_identity_sampler()),
    ('classifier', LogisticRegressionEstimator()),
    ]
    ,memory='./cache')
# NOTE: memory='./cache' removed intentionally. A stale joblib cache with
# n_jobs>1 was a prime suspect for the NaN scores. Re-enable later if needed.

# Sanity guard — the original NaN cascade was caused by sklearn treating the
# wrapped classifiers as regressors, which made every roc_auc / pr_auc scorer
# raise inside CV. Catch the misconfiguration here, before a 1000-fit sweep.
assert is_classifier(pipeline), (
    'Pipeline is not recognized as a classifier. Check that the final-step '
    'estimator inherits as (ClassifierMixin, BaseEstimator) — mixin first.'
)
pipeline

,steps,"[('imputer', ...), ('tamer', ...), ...]"
,transform_input,None
,memory,'./cache'
,verbose,False
,detector,'iforest'
,remediation,'winsorize'
,contamination,0.05
,z_threshold,3.0
,mad_threshold,3.5
,lower_quantile,0.01
,upper_quantile,0.99


## Search space
Each sub-dict pins one classifier and lists compatible step choices + hyper-parameters. `RandomizedSearchCV` samples 200 combinations from the cross-product of all sub-dicts.

In [4]:
from build_param_grid import build_param_grid, diversity_report

TOTAL_N = 200   # candidates pre-sampled via Latin Hypercube + quotas
param_grid, family_index = build_param_grid(spw_base, total_n=TOTAL_N, seed=42)
print(f'Pre-sampled {len(param_grid)} candidates across {len(set(family_index))} model families.')


Pre-sampled 200 candidates across 6 model families.


In [5]:
# Diversity audit — entropy across categorical axes and spread across
# continuous axes, computed BEFORE any fitting. Flags axes with H<0.80.
_ = diversity_report(param_grid, family_index)


=== Family distribution ===
  catboost     20   (10.0%)
  lgbm         60   (30.0%)
  lr           20   (10.0%)
  rf           20   (10.0%)
  stacker      20   (10.0%)
  xgb          60   (30.0%)

=== Categorical entropy (Hₙ ∈ [0,1]; 1.0 = uniform across categories) ===
  _family                              H=0.917   k= 6   n=200
  imputer                              H=1.000   k= 5   n=200
  normalizer                           H=1.000   k= 2   n=200
  tamer__detector                      H=1.000   k= 3   n=200
  tamer__remediation                   H=1.000   k= 6   n=200
  selector                             H=1.000   k= 5   n=200
  balancer                             H=1.000   k= 4   n=200
  classifier__max_features             H=0.998   k= 3   n= 20
  classifier__penalty                  H=1.000   k= 2   n= 20

=== Continuous spread (per-axis; only over candidates that sample it) ===
  axis                                                  min         max        std     n
  class

## Fit the search

In [6]:
import traceback
from sklearn.model_selection import cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------------------------------------------------------------------------
# Stage 1: smoke test on the default pipeline (no grid overrides).
# Confirms imputer → tamer → normalizer → selector → balancer → classifier
# wiring before spending real compute on the full sweep.
# ---------------------------------------------------------------------------
# print('--- smoke test (default pipeline, error_score=raise) ---')
# try:
#     smoke = cross_validate(
#         pipeline, x_train, y_train,
#         scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
#         cv=cv, n_jobs=1, error_score='raise',
#         return_train_score=False,
#     )
#     print(f"  pr_auc  mean: {np.mean(smoke['test_pr_auc']):.4f}")
#     print(f"  roc_auc mean: {np.mean(smoke['test_roc_auc']):.4f}")
#     smoke_ok = True
# except Exception:
#     print('SMOKE TEST FAILED — do not run the full search yet.')
#     traceback.print_exc()
#     smoke_ok = False

# ---------------------------------------------------------------------------
# Stage 2: GridSearchCV over the pre-sampled candidates.
# Each dict in `param_grid` already pins one full configuration (every value
# wrapped in a length-1 list), so GridSearchCV evaluates each one exactly once.
# error_score=np.nan keeps pathological combos (e.g. SMOTE k_neighbors >
# minority size in a fold) from killing the whole sweep; Stage 3 surfaces them.
# ---------------------------------------------------------------------------
smoke_ok=True
search = None
if smoke_ok:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc'},
        refit='pr_auc',
        cv=cv,
        n_jobs=-1,
        verbose=2,
        return_train_score=False,
        error_score=np.nan,
    )
    try:
        search.fit(x_train, y_train)
    except Exception:
        print('GridSearchCV.fit raised — full traceback below.')
        traceback.print_exc()
        search = None

# ---------------------------------------------------------------------------
# Stage 3: surface silent failures (NaN scores) inside cv_results_, and report
# PR-AUC dispersion — the metric that tells you whether the wider search
# actually produced wider results.
# ---------------------------------------------------------------------------
if search is not None and hasattr(search, 'cv_results_'):
    res = pd.DataFrame(search.cv_results_)
    n_total  = len(res)
    n_failed = int(res['mean_test_pr_auc'].isna().sum())
    print(f'\n--- post-search audit ---')
    print(f'configs total : {n_total}')
    print(f'configs OK    : {n_total - n_failed}')
    print(f'configs NaN   : {n_failed}')
    if n_failed and 'fit_error' in res.columns:
        msgs = (
            res.loc[res['mean_test_pr_auc'].isna(), 'fit_error']
               .dropna().astype(str).str.slice(0, 240).value_counts().head(5)
        )
        if len(msgs):
            print('\nTop failure messages (truncated):')
            for msg, n in msgs.items():
                print(f'  [{n}x] {msg}')

    pr = pd.to_numeric(res['mean_test_pr_auc'], errors='coerce').dropna()
    if len(pr):
        print(f'\n--- PR-AUC dispersion across candidates ---')
        print(f'  min    : {pr.min():.4f}')
        print(f'  p25    : {pr.quantile(0.25):.4f}')
        print(f'  median : {pr.median():.4f}')
        print(f'  p75    : {pr.quantile(0.75):.4f}')
        print(f'  max    : {pr.max():.4f}')
        print(f'  std    : {pr.std():.4f}    range: {pr.max() - pr.min():.4f}')


Fitting 5 folds for each of 200 candidates, totalling 1000 fits


/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'many

...


[CV] END balancer=SMOTESampler(), classifier=CalibratedClassifierCV(cv=3,
                       estimator=StackingClassifier(estimators=[('xgb',
                                                                 XGBoostEstimator(colsample_bytree=0.9,
                                                                                  n_estimators=300,
                                                                                  subsample=0.9)),
                                                                ('lgbm',
                                                                 LightGBMEstimator(learning_rate=0.05,
                                                                                   n_estimators=300,
                                                                                   reg_lambda=1.0)),
                                                                ('lr',
                                                                 LogisticRegressionEstimator())],
   


[CV] END balancer=BorderlineSMOTESampler(), classifier=CalibratedClassifierCV(cv=3,
                       estimator=StackingClassifier(estimators=[('xgb',
                                                                 XGBoostEstimator(colsample_bytree=0.9,
                                                                                  n_estimators=300,
                                                                                  subsample=0.9)),
                                                                ('lgbm',
                                                                 LightGBMEstimator(learning_rate=0.05,
                                                                                   n_estimators=300,
                                                                                   reg_lambda=1.0)),
                                                                ('lr',
                                                                 LogisticRegressionEstimato


[CV] END balancer=SMOTESampler(k_neighbors=5), classifier=LogisticRegressionEstimator(), classifier__C=0.001039903674570102, classifier__penalty=l2, imputer=ClassMeanImputer(), normalizer=ZScoreNormalizationNorm(), selector=TreeBasedSelection(), tamer__contamination=0.055470175634693704, tamer__detector=statistical, tamer__mad_threshold=2.323167576158225, tamer__remediation=cluster_local_median; total time=   1.8s
[CV] END balancer=SMOTESampler(k_neighbors=5), classifier=CalibratedClassifierCV(cv=3,
                       estimator=StackingClassifier(estimators=[('xgb',
                                                                 XGBoostEstimator(colsample_bytree=0.9,
                                                                                  n_estimators=300,
                                                                                  subsample=0.9)),
                                                                ('lgbm',
                                              


[CV] END balancer=BorderlineSMOTESampler(), classifier=CalibratedClassifierCV(cv=3,
                       estimator=StackingClassifier(estimators=[('xgb',
                                                                 XGBoostEstimator(colsample_bytree=0.9,
                                                                                  n_estimators=300,
                                                                                  subsample=0.9)),
                                                                ('lgbm',
                                                                 LightGBMEstimator(learning_rate=0.05,
                                                                                   n_estimators=300,
                                                                                   reg_lambda=1.0)),
                                                                ('lr',
                                                                 LogisticRegressionEstimato


--- post-search audit ---
configs total : 200
configs OK    : 200
configs NaN   : 0

--- PR-AUC dispersion across candidates ---
  min    : 0.1642
  p25    : 0.2100
  median : 0.2283
  p75    : 0.2435
  max    : 0.2838
  std    : 0.0258    range: 0.1196


## Inspect the winner

In [7]:
def _bail(msg):
    print(f'[skipped] {msg}')

# Guard: only run if the search actually fitted.
if search is None or not hasattr(search, 'best_estimator_'):
    _bail('No fitted search available. Fix errors reported in the previous cell and re-run.')
else:
    print(f'Best CV PR-AUC : {search.best_score_:.4f}')
    print('Best pipeline   :')
    for name, step in search.best_estimator_.named_steps.items():
        label = 'passthrough' if isinstance(step, str) else type(step).__name__
        print(f'  {name:11s} -> {label}')

    print('\nBest params:')
    for k, v in search.best_params_.items():
        print(f'  {k}: {v}')

    # Hold-out scores at the *default* 0.5 threshold (kept for continuity).
    y_proba = search.predict_proba(x_val)[:, 1]
    y_pred_default = (y_proba >= 0.5).astype(int)

    val_roc_auc = roc_auc_score(y_val, y_proba)
    val_pr_auc  = average_precision_score(y_val, y_proba)

    print(f'\nHold-out ROC-AUC : {val_roc_auc:.4f}')
    print(f'Hold-out PR-AUC  : {val_pr_auc:.4f}')
    print('\nConfusion matrix @ threshold=0.50 (uninformative on imbalanced data):')
    print(confusion_matrix(y_val, y_pred_default))
    print(classification_report(y_val, y_pred_default, zero_division=0))

    # ----------------------------------------------------------------------
    # Post-hoc threshold optimization.
    # A model that predicts at 0.5 on 11% prevalence is structurally biased
    # toward the majority class. We pick the threshold that maximizes F1 on
    # the holdout's precision-recall curve (swap to F2 if recall matters more
    # clinically).
    # ----------------------------------------------------------------------
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
    f1s = 2 * precisions[:-1] * recalls[:-1] / np.clip(precisions[:-1] + recalls[:-1], 1e-12, None)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[best_idx])

    beta = 2.0
    f2s = (1 + beta**2) * precisions[:-1] * recalls[:-1] / np.clip(beta**2 * precisions[:-1] + recalls[:-1], 1e-12, None)
    best_f2_idx = int(np.nanargmax(f2s))
    best_f2_thr = float(thresholds[best_f2_idx])

    fpr, tpr, roc_thr = roc_curve(y_val, y_proba)
    spec90_mask = (1 - fpr) >= 0.90
    if spec90_mask.any():
        idx90 = int(np.argmax(tpr * spec90_mask))
        recall_at_spec90 = float(tpr[idx90])
        thr_at_spec90    = float(roc_thr[idx90])
    else:
        recall_at_spec90, thr_at_spec90 = float('nan'), float('nan')

    print('\n--- Operating points ---')
    print(f'Best F1 threshold : {best_thr:.4f}   (F1={f1s[best_idx]:.3f}, P={precisions[best_idx]:.3f}, R={recalls[best_idx]:.3f})')
    print(f'Best F2 threshold : {best_f2_thr:.4f}  (F2={f2s[best_f2_idx]:.3f}, P={precisions[best_f2_idx]:.3f}, R={recalls[best_f2_idx]:.3f})')
    print(f'Recall @ Spec=0.90: {recall_at_spec90:.3f}  (threshold={thr_at_spec90:.4f})')

    y_pred_tuned = (y_proba >= best_thr).astype(int)
    print(f'\nConfusion matrix @ tuned F1 threshold={best_thr:.4f}:')
    print(confusion_matrix(y_val, y_pred_tuned))
    print(classification_report(y_val, y_pred_tuned, zero_division=0))

Best CV PR-AUC : 0.2838
Best pipeline   :
  imputer     -> KNNImputerWrapper
  tamer       -> OutlierTamer
  normalizer  -> RobustScalerNorm
  selector    -> TreeBasedSelection
  balancer    -> SMOTESampler
  classifier  -> LogisticRegressionEstimator

Best params:
  balancer: SMOTESampler(k_neighbors=5)
  classifier: LogisticRegressionEstimator()
  classifier__C: 0.06084106114127848
  classifier__penalty: l1
  imputer: KNNImputerWrapper()
  normalizer: RobustScalerNorm()
  selector: TreeBasedSelection()
  tamer__contamination: 0.010475392330538177
  tamer__detector: cluster
  tamer__mad_threshold: 3.1725433150864397
  tamer__remediation: percentile_clip



Hold-out ROC-AUC : 0.7033
Hold-out PR-AUC  : 0.2527

Confusion matrix @ threshold=0.50 (uninformative on imbalanced data):
[[933 387]
 [ 74  93]]
              precision    recall  f1-score   support

           0       0.93      0.71      0.80      1320
           1       0.19      0.56      0.29       167

    accuracy                           0.69      1487
   macro avg       0.56      0.63      0.54      1487
weighted avg       0.84      0.69      0.74      1487


--- Operating points ---
Best F1 threshold : 0.6539   (F1=0.308, P=0.258, R=0.383)
Best F2 threshold : 0.3665  (F2=0.463, P=0.170, R=0.814)
Recall @ Spec=0.90: 0.275  (threshold=0.7100)

Confusion matrix @ tuned F1 threshold=0.6539:
[[1136  184]
 [ 103   64]]
              precision    recall  f1-score   support

           0       0.92      0.86      0.89      1320
           1       0.26      0.38      0.31       167

    accuracy                           0.81      1487
   macro avg       0.59      0.62      0.60    

## Leaderboard

In [8]:
if search is None or not hasattr(search, 'cv_results_'):
    print('[skipped] No cv_results_ available (fit failed).')
    cv_df = None
else:
    cv_df = (
        pd.DataFrame(search.cv_results_)
          .sort_values('mean_test_pr_auc', ascending=False)
          [['mean_test_pr_auc', 'std_test_pr_auc', 'mean_test_roc_auc', 'std_test_roc_auc', 'params']]
          .head(15)
          .reset_index(drop=True)
    )
    pd.DataFrame(search.cv_results_).to_csv("all_fits.csv",index=False)
cv_df

,mean_test_pr_auc,std_test_pr_auc,mean_test_roc_auc,std_test_roc_auc,params
0,0.283799,0.015297,0.736191,0.011608,"{'balancer': SMOTESampler(k_neighbors=5), 'cla..."
1,0.282230,0.021837,0.736228,0.012435,"{'balancer': SMOTESampler(), 'classifier': Log..."
2,0.281461,0.020159,0.736361,0.010467,{'balancer': FunctionSampler(func=<function _i...
3,0.278473,0.023057,0.734361,0.010561,{'balancer': FunctionSampler(func=<function _i...
4,0.276528,0.020603,0.730023,0.008931,"{'balancer': SMOTESampler(k_neighbors=5), 'cla..."
5,0.276175,0.016936,0.733995,0.006427,{'balancer': FunctionSampler(func=<function _i...
6,0.276091,0.018264,0.735442,0.011528,{'balancer': FunctionSampler(func=<function _i...
7,0.274506,0.024139,0.732254,0.011423,"{'balancer': SMOTESampler(), 'classifier': Log..."
8,0.274466,0.014631,0.726722,0.013088,"{'balancer': BorderlineSMOTESampler(), 'classi..."
9,0.274248,0.017676,0.732887,0.010685,"{'balancer': SMOTESampler(), 'classifier': Log..."


## Persist artifacts

In [9]:
if search is None or not hasattr(search, 'best_estimator_'):
    print('[skipped] No fitted estimator to persist.')
else:
    out_dir = os.path.dirname(path) or '.'
    os.makedirs(out_dir, exist_ok=True)

    joblib.dump(search.best_estimator_, os.path.join(out_dir, 'pipeline.pkl'))
    joblib.dump(all_mappings,           os.path.join(out_dir, 'all_mapping.pkl'))
    joblib.dump(y_mappings,             os.path.join(out_dir, 'y_mappings.pkl'))
    joblib.dump(list(X.columns),        os.path.join(out_dir, 'features.pkl'))

    # Persist the tuned operating point — required for inference, since the
    # default 0.5 is the wrong cut for this prevalence.
    joblib.dump(
        {
            'threshold_f1':     best_thr,
            'threshold_f2':     best_f2_thr,
            'threshold_spec90': thr_at_spec90,
            'val_roc_auc':      val_roc_auc,
            'val_pr_auc':       val_pr_auc,
            'recall_at_spec90': recall_at_spec90,
        },
        os.path.join(out_dir, 'operating_point.pkl'),
    )

    print('Saved:', os.path.join(out_dir, 'pipeline.pkl'))
    print('Saved:', os.path.join(out_dir, 'operating_point.pkl'))

Saved: ./Final/pipeline.pkl
Saved: ./Final/operating_point.pkl
